In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision.models as models
from torch.amp import autocast, GradScaler
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
transform_train = transforms.Compose([

    transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
    transforms.Resize((64,64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),

    transforms.ToTensor(),
    transforms.RandomErasing(p=0.3),

    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

In [ ]:
train_data = datasets.ImageFolder(
    root="/kaggle/input/datasets/jxyle1/emotiondatav2/emotiondata/train",
    transform=transform_train
)


test_data = datasets.ImageFolder(
    root="/kaggle/input/datasets/jxyle1/emotiondatav2/emotiondata/test",
    transform=transform_test
)

In [ ]:
train_dataload = DataLoader(
    train_data,batch_size=64,shuffle=True,num_workers=2, pin_memory=True
)
test_dataload = DataLoader(
    test_data,batch_size=64,shuffle=False,num_workers=2,pin_memory=True
)

In [ ]:
class block(nn.Module):
    def __init__(self,in_channels,out_channels,identity_downsample=None,stride=1):
        super(block,self).__init__()
        self.expansion = 1
        
        self.conv1 = nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=stride,padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)


        self.relu = nn.ReLU()
        self.identity_downsample = identity_downsample

    def forward(self,x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        #if we need to change the shape
        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)

        x = x + identity
        x = self.relu(x)
        return x

In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, layers, image_channels,num_classes):
        super(ResNet, self).__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()

        # ResNet layers
        self.layer1 = self._make_layer(block, layers[0], out_channels=64, stride=1)
        self.layer2 = self._make_layer(block, layers[1], out_channels=128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], out_channels=256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], out_channels=512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fcl = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(512,6)

        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fcl(x)

        return x

    def _make_layer(self, block, num_residual_block, out_channels, stride):
        identity_downsample = None
        layers = []

        if stride != 1 or self.in_channels != out_channels:
            identity_downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )
        
        layers.append(block(self.in_channels, out_channels, identity_downsample, stride))
        self.in_channels = out_channels

        for _ in range(num_residual_block - 1):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

In [ ]:
import numpy as np
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

In [ ]:
max_epochs = 100
net = Resnet18().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(net.parameters(),lr = 3e-4,weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max_epochs)

In [ ]:
scaler = GradScaler()
for epoch in range(max_epochs):
    print(f"Training Epoch {epoch}...")

    running_loss = 0.0

    for i,data in enumerate(train_dataload):
        inputs, labels = data
        inputs,labels = inputs.to(device),labels.to(device)
        optimizer.zero_grad()

        inputs, labels_a, labels_b, lam = mixup_data(inputs, labels)
        
        with autocast(device_type=device):
            outputs = net(inputs)
            loss = lam * loss_function(outputs, labels_a) + (1 - lam) * loss_function(outputs, labels_b)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
    epoch_loss = running_loss / len(train_dataload)
    scheduler.step()
    print(f"Loss: {running_loss/len(train_dataload):.2f}")

In [ ]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_dataload:
        images,labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _,predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy}%")

In [ ]:
import joblib

model_data = {
    "model_state_dict" : net.state_dict(),
    "class_to_idx": train_data.class_to_idx
}
import torch

torch.save(model_data, "/kaggle/working/emotiondetecter_model.pth")